## Shadow model for cloud cover parametrization using piecewise affine linear quasi interpolant on the full grid

basis function vgl doi 10.1017/S0962492904000182

Remark: For efficency, we do not compute the entire interpolant and evaluate it for each data point. Instead, for each data point, we locate the cell in the grid it is in, and then compute the interpolant only on the cell and in particular evaluate the QML model on the nodes of the grid cell.
In the analytical setting, without shot noise, this is equvialent to evaluating the entire grid from which we compute the interpolant.
When incorporating shot noise, this means that for two points in the same cell, or cells sharing a node, we use two different interpolants.
Since we only evaluate the shadow model by the MSE of the data points (independent) of each other, this does not distort the results.


In [ ]:

import sys
import os
import importlib
from sklearn.metrics import mean_squared_error
import quasi_interpolation as qi
import pennylane as qml
import pennylane.numpy as np
import jax
from jax import numpy as jnp
from pathlib import Path



jax.config.update("jax_enable_x64", True)

path_base = Path(os.getcwd()) 
# Current path for importing custom functions
sys.path.insert(0, str(path_base / "clc_functions")) 

import input_transform
importlib.reload(input_transform)
from input_transform import inputs_transform, inverse_transform_clc

import qnn_layouts_pennylane
importlib.reload(qnn_layouts_pennylane)
import qnn_layouts_pennylane as pqcs

# Folder in which to find the test inputs
test_inputs_folder = 'test_data/'

transform_output = True
name_out_transf = ''
if transform_output:
    name_out_transf = '_transformedCLC'

### Interval within which transformed clc should be bounded
bound_output = [0.0, 1.0]

### Upper bound for input transformation
transform_input = True
upperbound = np.pi
upperbound_name = '1p0pi'

batch_size = 100
n_batch_name = str(batch_size)

# Learning rate
learning_rate = 0.001
learning_rate_name = '0p001'

### Kept features
features_kept = ['hus', 'clw', 'cli', 'ta', 'pa', 'hwind']
no_of_features = len(features_kept)

### Architecture specifications
no_qubits = no_of_features


## Load QML Model (L. Pastoris Code) and optimal parameters

In [ ]:

### PQC architecture layout
# No of shots for circuit evaluation
no_shots = 100 # or #inf
#No of shots that were used in training 
no_shots_training = 'inf'
#Architecture to be used 'ZZXY' or 'XYZ'
name_arch = 'XYZ'
### PQC architecture layout + optimal params
if name_arch == 'XYZ':
    pqc_layout = pqcs.XYZ_circuit;  name_arch = 'XYZ';  n_enc = 4;  n_dec = 2; 
    if no_shots_training == 'inf':
        best_exp = 1 
    else:
        best_exp = 4
elif name_arch == 'ZZXY':
    pqc_layout = pqcs.ZZXY_circuit;  name_arch = 'ZZXY';  n_enc = 2;  n_dec = 5; # best_exp = 6 # for inifinite shots
    if no_shots_training == 'inf':
        best_exp = 6 
    else:
        best_exp = 3

n_enc_name = str(n_enc)
n_dec_name = str(n_dec)

# Load optimal params:
if no_shots_training == 'inf':
    params_folder = 'optimal_params/'
    namefilepars = 'optimal_params'
    name_end = ('_upperbound' + upperbound_name + name_out_transf + '_' + name_arch + '_Nenc' + n_enc_name + 
                '_Ndec' + n_dec_name + '_batch' + n_batch_name + '_lr' + learning_rate_name + '_test' + str(best_exp))
    filename_pars = namefilepars + name_end + '.npy'
else: #varreg, trainign with 1000t
    params_folder = 'optimal_params/'
    filename_pars = 'optimal_params_Nshots'+str(no_shots_training)+'_multinomial_transformedCLC_' + name_arch + '_Nenc' +str(n_enc) + '_Ndec' + str(n_dec) + '_batch100_alpha0p005_iniparam12345_test' + str(best_exp) +'.npy'
    

path_file = os.path.join(params_folder, filename_pars)
opt_params_np = np.load(path_file)
opt_params = jnp.asarray(opt_params_np)


## Load test data


In [ ]:
ind_features = [0,1,2,3,4,6]
### ---------------------------------------------------------------------------------------- ###
## ----------------------------------- Load testing data ------------------------------------ ##
### ---------------------------------------------------------------------------------------- ###

namefilein = 'cirrus_inputs_raw_8features.npy'
namefileout = 'cirrus_outputs_raw_8features.npy'
path_file = os.path.join(test_inputs_folder, namefilein)
test_inputs_cirrus_full = np.load(path_file)
test_inputs_cirrus = test_inputs_cirrus_full[:,ind_features]
path_file = os.path.join(test_inputs_folder, namefileout)
test_outputs_cirrus = np.load(path_file)
no_testing_data_cirrus = test_inputs_cirrus.shape[0]
### Get indices of samples with very low condensate, which are the
### the ones removed from the training data, since for them clc=0 !!!
clw = test_inputs_cirrus[:, np.where(np.array([k=='clw' for k in features_kept]))[0][0]]
cli = test_inputs_cirrus[:, np.where(np.array([k=='cli' for k in features_kept]))[0][0]]
clt = clw + cli
IIIlowclt = (clt <= 1.0e-08)
III_cirrus = np.squeeze(np.logical_not(IIIlowclt))
indsIII_cirrus = np.where(III_cirrus)[0]
no_test_samples_to_evaluate_cirrus = np.sum(III_cirrus)
print('No. test samples to evaluate (cirrus): ', no_test_samples_to_evaluate_cirrus)

namefilein = 'cumulus_inputs_raw_8features.npy'
namefileout = 'cumulus_outputs_raw_8features.npy'
path_file = os.path.join(test_inputs_folder, namefilein)
test_inputs_cumulus_full = np.load(path_file)
test_inputs_cumulus = test_inputs_cumulus_full[:,ind_features]
path_file = os.path.join(test_inputs_folder, namefileout)
test_outputs_cumulus = np.load(path_file)
no_testing_data_cumulus = test_inputs_cumulus.shape[0]
### Get indices of samples with very low condensate, which are the
### the ones removed from the training data, since for them clc=0 !!!
clw = test_inputs_cumulus[:, np.where(np.array([k=='clw' for k in features_kept]))[0][0]]
cli = test_inputs_cumulus[:, np.where(np.array([k=='cli' for k in features_kept]))[0][0]]
clt = clw + cli
IIIlowclt = (clt <= 1.0e-08)
III_cumulus = np.squeeze(np.logical_not(IIIlowclt))
indsIII_cumulus = np.where(III_cumulus)[0]
no_test_samples_to_evaluate_cumulus = np.sum(III_cumulus)
print('No. test samples to evaluate (cumulus): ', no_test_samples_to_evaluate_cumulus)

namefilein = 'deepconv_inputs_raw_8features.npy'
namefileout = 'deepconv_outputs_raw_8features.npy'
path_file = os.path.join(test_inputs_folder, namefilein)
test_inputs_deepconv_full = np.load(path_file)
test_inputs_deepconv = test_inputs_deepconv_full[:,ind_features]
path_file = os.path.join(test_inputs_folder, namefileout)
test_outputs_deepconv = np.load(path_file)
no_testing_data_deepconv = test_inputs_deepconv.shape[0]
### Get indices of samples with very low condensate, which are the
### the ones removed from the training data, since for them clc=0 !!!
clw = test_inputs_deepconv[:, np.where(np.array([k=='clw' for k in features_kept]))[0][0]]
cli = test_inputs_deepconv[:, np.where(np.array([k=='cli' for k in features_kept]))[0][0]]
clt = clw + cli
IIIlowclt = (clt <= 1.0e-08)
III_deepconv = np.squeeze(np.logical_not(IIIlowclt))
indsIII_deepconv = np.where(III_deepconv)[0]
no_test_samples_to_evaluate_deepconv = np.sum(III_deepconv)
print('No. test samples to evaluate (deepconv): ', no_test_samples_to_evaluate_deepconv)

namefilein = 'stratus_inputs_raw_8features.npy'
namefileout = 'stratus_outputs_raw_8features.npy'
path_file = os.path.join(test_inputs_folder, namefilein)
test_inputs_stratus_full = np.load(path_file)
test_inputs_stratus = test_inputs_stratus_full[:,ind_features]
path_file = os.path.join(test_inputs_folder, namefileout)
test_outputs_stratus = np.load(path_file)
no_testing_data_stratus = test_inputs_stratus.shape[0]
### Get indices of samples with very low condensate, which are the
### the ones removed from the training data, since for them clc=0 !!!
clw = test_inputs_stratus[:, np.where(np.array([k=='clw' for k in features_kept]))[0][0]]
cli = test_inputs_stratus[:, np.where(np.array([k=='cli' for k in features_kept]))[0][0]]
clt = clw + cli
IIIlowclt = (clt <= 1.0e-08)
III_stratus = np.squeeze(np.logical_not(IIIlowclt))
indsIII_stratus = np.where(III_stratus)[0]
no_test_samples_to_evaluate_stratus = np.sum(III_stratus)
print('No. test samples to evaluate (stratus): ', no_test_samples_to_evaluate_stratus)

### Transform inputs if needed
if transform_input:
    bound_input = [0.0, upperbound]
    bounds = [bound_input for _ in features_kept]
    test_inputs_cirrus_t = inputs_transform(test_inputs_cirrus, features_kept, bounds)
    test_inputs_cumulus_t = inputs_transform(test_inputs_cumulus, features_kept, bounds)
    test_inputs_deepconv_t = inputs_transform(test_inputs_deepconv, features_kept, bounds)
    test_inputs_stratus_t = inputs_transform(test_inputs_stratus, features_kept, bounds)

### Convert testing data to jax numpy arrays
jnp_test_inputs_cirrus = jnp.asarray(test_inputs_cirrus_t)
jnp_test_inputs_cumulus = jnp.asarray(test_inputs_cumulus_t)
jnp_test_inputs_deepconv = jnp.asarray(test_inputs_deepconv_t)
jnp_test_inputs_stratus = jnp.asarray(test_inputs_stratus_t)

## Reduce data set to first 1000 points for each cloud type

test_data = np.concatenate([jnp_test_inputs_cirrus[indsIII_cirrus[0:1000],:], jnp_test_inputs_cumulus[indsIII_cumulus[0:1000],:],jnp_test_inputs_stratus[indsIII_stratus[0:1000],:],jnp_test_inputs_deepconv[indsIII_deepconv[0:1000],:]], axis = 0)
test_output =  np.concatenate([test_outputs_cirrus[indsIII_cirrus[0:1000]], test_outputs_cumulus[indsIII_cumulus[0:1000]],test_outputs_stratus[indsIII_stratus[0:1000]],test_outputs_deepconv[indsIII_deepconv[0:1000]]], axis = 0)




 


In [ ]:
### ---------------------------------------------------------------------------------------- ###
## ---------------------------------- Initialize QNN model ---------------------------------- ##
### ---------------------------------------------------------------------------------------- ###

no_gate_angles = pqcs.no_of_angles_pqc(name_arch, no_qubits, n_enc, n_dec)
no_params = no_gate_angles + no_qubits + 1

### Define the quantum device
if no_shots == 'inf':
    dev = qml.device('default.qubit.jax', wires=no_qubits)
else:
    dev = qml.device('default.qubit.jax', wires=no_qubits, shots=no_shots)

### Define pqc with measured observables
@qml.qnode(dev, interface="jax")
def qnn_pqc(inputs, pars):
    pqc_layout(inputs, pars, n_enc=n_enc, n_dec=n_dec, wires=dev.wires)
    return [qml.expval(qml.PauliZ(i)) for i in range(no_qubits)]

### Define the QNN model (pqc + postprocessing)
@jax.jit
def model_qnn(params, inputs):
    # Split paramter vector in the different components 
    no_angles = no_gate_angles
    no_weights = no_qubits
    no_bias = 1
    angles = jax.lax.dynamic_slice(params, [0], [no_angles])
    weights = jax.lax.dynamic_slice(params, [no_angles], [no_weights])
    bias = jax.lax.dynamic_slice(params, [no_angles + no_weights], [no_bias])

    # Computation of the quantum circuit in 'measured_batches'
    # 'measured_batches' contains a no_qubits-long list, 
    # where the i-th element is the jnp.array containing the 
    # Z(i) expectation value over the input batch
    measured_batches = qnn_pqc(inputs, angles)

    # Classical post-processing (weighted avg. + bias)
    weighted_output = weights[0] * measured_batches[0]
    for i in range(1,no_qubits):
        weighted_output = weighted_output + weights[i] * measured_batches[i]
    predictions = weighted_output + bias
    return jnp.squeeze(predictions)

In [ ]:
data_n = {'arch': name_arch, 'Path params':  filename_pars, 'shots': no_shots, 'shots_training': no_shots_training}

In [ ]:
def post_processing(outsbatch):
    outsbatch = np.asarray(outsbatch)
    outsbatch = np.squeeze(outsbatch)
    outsbatch = np.minimum(outsbatch, 1.0)
    outsbatch = np.maximum(outsbatch, 0.0)
    outsbatch = inverse_transform_clc(outsbatch)
    return outsbatch
    

#@jax.jit
def eval_model(data):
    predictions = jnp.array(model_qnn(opt_params,data))
    return predictions



## Evaluate Interpolant on test data set

In [ ]:
#Generate data (full grid)
d = no_qubits
#refinement parameters (2*np.pi*2^(-J)*j for j in 0,...2^(J)
ref = range(3,7) # refinement 
mse = np.zeros((len(ref),))
for index,L in enumerate(ref):
    print(L)
    J = [L]*d
    f_interpolant = lambda x,J = J,d = d: qi.local_interpolant(x,eval_model,J,d)
    batch_size =4000
    pred_test_outputs_cs= np.zeros(4000)
    ins_batch = test_data
    outs_batch= f_interpolant(ins_batch)
    outs_batch = post_processing(outs_batch)
   
    pred_test_outputs_cs= np.squeeze(outs_batch)
    mse[L-3] = mean_squared_error(test_output,pred_test_outputs_cs)
    


## Save data

In [ ]:

data_n['mse'] = mse
data_n[ 'L'] = range(3,8)
data_n['setting'] = 'pw affine linear interpolant on full grid, first 1000 points of each test data set'
np.save('result_full_qi_'+name_arch +str(no_shots_training) +'t_'+ str(no_shots) +'s_4000_7',data_n)
print(mse)